# DM_G4_P0006_integer_binary_fixedcharge_answer

## 0. 정답본 범위
- Gate: G4
- Phase: P0006
- Topic: 정수계획, 0-1 선택변수, 고정비용, 논리 제약
- Problem notebook: DM_G4_P0006_integer_binary_fixedcharge.ipynb
- Correct answers included: Yes
- Output language: Korean

## 1. 핵심 기준표
| 기준 | 정답 판단 |
|---|---|
| 0-1 변수 | 선택, 설치, 개방 여부처럼 스위치인 의사결정이다. |
| 연속 변수 | 생산량, 배송량, 수송량처럼 나누어 배정될 수 있는 수량이다. |
| LP relaxation | 정수조건 또는 binary 조건을 제거해 연속 범위로 푼 완화 문제다. |
| fixed-charge | 수량 `x`와 개방 `y`를 분리하고 `x <= U y`, 필요 시 `L y <= x`로 연결한다. |
| 반올림 금지 | 예산, prerequisite, equality, capacity 같은 제약을 깨뜨릴 수 있으므로 정수해 증명이 아니다. |

## 2. 문항별 정답

### 문제 1 정답 — 탄소저감 R&D 포트폴리오
**변수와 목적함수**
- `x_j = 1`이면 프로젝트 `j` 선정, `0`이면 미선정, `j=1,...,8`.
- Maximize `14x1+11x2+10x3+13x4+9x5+12x6+8x7+7x8`.

**예산 제약**
- `3x1+2x2+2x3+4x4+x5+2x6+x7+x8 <= 8`
- `2x1+3x2+x3+2x4+3x5+2x6+2x7+x8 <= 9`
- `2x1+x2+3x3+2x5+3x6+2x7+2x8 <= 8`

**정책 제약**
- 배터리 최대 2개: `x1+x2+x3 <= 2`
- 프로젝트 5의 prerequisite: `x5 <= x4`
- 프로젝트 6과 7 중 정확히 하나: `x6+x7 = 1`
- 프로젝트 2와 8 all-or-none: `x2 = x8`
- 전체 개수: `3 <= x1+...+x8 <= 5`
- domain: `x_j in {0,1}`.

**LP relaxation과 반올림 진단**
- relaxation에서는 `x_j in {0,1}` 대신 `0 <= x_j <= 1`을 둔다.
- 제시 후보는 `x2`, `x5`, `x8`이 소수이므로 정수해가 아니다.
- 0.5를 1로 올리는 반올림을 하면 `x1+x2+x3=3`이 되어 배터리 최대 2개 조건을 위반하고, 1년차 예산도 크게 초과한다.
- 따라서 relaxation 해는 bound 또는 진단용이지 최종 선택 조합이 아니다.

**검산 기준 최적 정수해 예시**
- 제공 데이터 기준 최적 선택은 `x=(1,1,0,0,0,1,0,1)`이고 가치 합은 `44`이다.
- 예산 사용량은 `(8,8,8)`이며 모든 정책 제약을 만족한다.

**Solver mapping**
- changing cells: `x1:x8` 선택 변수 영역.
- 목표셀: 가치 합계 셀, maximize.
- constraint cells: 연도별 예산 사용량, 정책 조건 좌변, 전체 선정 수.
- RHS cells: `8, 9, 8, 2, 1, 0, 3, 5` 등 제약 우변.
- bin option: `x1:x8` 전체에 binary 지정.

**source anchor / node_id**
- anchors: `DM_PDF04:p001:L012`, `DM_PDF04:p003:L004`, `DM_PDF04:p005:L002`, `DM_PDF04:p005:L006`, `DM_PDF04:p014:L002`, `DM_PDF04:p019:L002`, `DM_PDF04:p021:L002`
- node_id: `n_DM_PDF04.binary_variable`, `n_DM_PDF04.capital_budgeting`, `n_DM_PDF04.policy_logic_constraints`, `n_DM_PDF04.lp_relaxation`, `n_DM_PDF04.integer_solver_option`

### 문제 2 정답 — 도심 드론 배송 허브 운영
**변수와 목적함수**
- `x_j`: 허브 `j`의 하루 배송량, continuous, `x_j >= 0`.
- `y_j`: 허브 `j`를 열면 1, 닫으면 0, binary.
- Minimize `18x1+22x2+16x3+420y1+260y2+500y3`.

**제약식**
- 수요 충족: `x1+x2+x3 = 130`
- 상한 연결: `x1 <= 80y1`, `x2 <= 70y2`, `x3 <= 90y3`
- 최소 운영량: `20y1 <= x1`, `15y2 <= x2`, `30y3 <= x3`
- 상호배타: `y1+y3 <= 1`
- 허브 3 prerequisite: `y3 <= y2`
- 운영 허브 수: `y1+y2+y3 <= 2`
- domain: `x_j >= 0`, `y_j in {0,1}`.

**검산 기준 최적해 예시**
- `y=(0,1,1)`, `x=(0,40,90)`, 총비용 `3080`.
- 허브 3은 단위 처리비가 낮지만 prerequisite 때문에 허브 2도 함께 열려야 한다.

**오답 진단**
- `IF(x_j>0, fixed_j, 0)`는 목표셀에 비선형/불연속 논리를 숨긴다. 선형 정수계획에서는 `y_j`를 두고 고정비를 `fixed_j*y_j`로 넣어야 한다.
- `x_j <= U_j y_j`를 빼면 `y_j=0`인데도 `x_j>0`인 해가 가능해져 고정비를 내지 않고 배송하는 비현실적 해가 생긴다.

**source anchor / node_id**
- anchors: `DM_PDF04:p022:L002`, `DM_PDF04:p025:L002`, `DM_PDF04:p026:L002`, `DM_PDF04:p030:L002`, `DM_PDF04:p032:L002`, `DM_PDF04:p033:L002`
- node_id: `n_DM_PDF04.fixed_charge_model`, `n_DM_PDF04.either_or_constraint`, `n_DM_PDF04.binary_variable`, `n_DM_PDF04.integer_solver_option`

### 문제 3 정답 — 신선식품 생산-창고 통합 설계
**변수**
- `x_ij`: 공장 `i`에서 창고 `j`로 보내는 수송량, continuous.
- `w_jk`: 창고 `j`에서 고객 `k`로 보내는 배송량, continuous.
- `y_i`: 공장 `i` 가동여부, binary.
- `z_j`: 창고 `j` 가동여부, binary.

**목적함수**
Minimize
`4x11+6x12+5x21+3x22`
`+ 7w11+4w12+6w13+5w21+6w22+4w23`
`+ 900y1+700y2+500z1+420z2`.

**제약식**
- 공장 용량: `x11+x12 <= 120y1`, `x21+x22 <= 100y2`
- 창고 용량: `w11+w12+w13 <= 130z1`, `w21+w22+w23 <= 120z2`
- 창고 flow balance: `x11+x21 = w11+w12+w13`, `x12+x22 = w21+w22+w23`
- 고객 수요: `w11+w21=70`, `w12+w22=80`, `w13+w23=60`
- 최소 하나 공장: `y1+y2 >= 1`
- 최소 하나 창고: `z1+z2 >= 1`
- 창고 2 prerequisite: `z2 <= y2`
- 패키지 계약: `y1 = z1`
- domain: `x,w >= 0`, `y,z in {0,1}`.

**모형 분류와 검산 기준 최적해 예시**
- 연속 수량변수와 binary 개방변수가 함께 있으므로 mixed integer 모형이다.
- 제공 데이터 기준으로 총수요 210을 처리하려면 두 공장과 두 창고가 모두 필요하다.
- 한 최적 흐름 예시는 `x11=90, x12=20, x21=0, x22=100`, `w11=0, w12=80, w13=10`, `w21=70, w22=0, w23=50`이고 총비용은 `4230`이다.
- 같은 총비용을 내는 흐름이 여러 개 있을 수 있으므로, 정답 평가는 formulation과 비용 검산을 함께 본다.

**오답 진단**
- 수송량이 양수이면 자동으로 열렸다고 해석하면 고정비가 목적함수에 들어가지 않는다.
- 개방 변수 없이 수송량만 두면 “열지 않았는데 수송한다” 또는 “고정비 없이 운영한다”는 해를 배제할 수 없다.

**source anchor / node_id**
- anchors: `DM_PDF04:p035:L002`, `DM_PDF04:p038:L002`, `DM_PDF04:p039:L002`, `DM_PDF06:p019:L002`
- node_id: `n_DM_PDF04.plant_warehouse_location_model`, `n_DM_PDF04.fixed_charge_model`, `n_DM_PDF04.binary_variable`, `n_DM_PDF06.transshipment_problem`

## 3. 채점 기준
| 항목 | 배점 | 부분점 기준 |
|---|---:|---|
| binary/continuous variable 분리 | 20 | `x`와 `y/z` 의미를 합치면 큰 감점 |
| objective와 constraints formulation | 25 | 비용/가치 항과 예산/수요/흐름 제약 누락 여부 확인 |
| policy/fixed-charge/either-or logic | 25 | prerequisite 방향, 상호배타, `x <= U y`, `L y <= x` 확인 |
| LP relaxation과 반올림 금지 설명 | 15 | 완화 domain과 반올림 infeasible 위험을 설명해야 함 |
| Solver mapping과 source anchor/node_id | 15 | 목표셀/변수셀/제약셀/RHS/bin 연결과 근거 표시 |

## 4. 오답튜터 기준표
| 오류 | 진단 질문 | 교정 힌트 |
|---|---|---|
| 0-1 변수에 비음조건만 넣음 | “이 값이 0.37이면 현실적으로 선택인가?” | binary 옵션 또는 `{0,1}` domain을 명시한다. |
| LP relaxation 해를 반올림함 | “반올림 후 모든 예산과 논리식이 유지되는가?” | relaxation은 bound이지 최종 정수해가 아니다. |
| 생산량 변수와 가동여부 변수를 합침 | “수량 40과 개방 1이 같은 단위인가?” | `x`는 수량, `y`는 스위치로 분리한다. |
| `x <= U y` 누락 | “닫힌 시설에서 양수 물량이 나올 수 있는가?” | 닫으면 `x=0`이 되도록 상한 연결식을 둔다. |
| `L y <= x` 누락 | “열었는데 최소 운영량 미만이어도 되는가?” | either-or 조건은 하한과 상한을 함께 둔다. |
| IF 함수로 고정비 처리 | “목표셀이 선형식인가?” | 고정비는 `fixed*y`로 선형화한다. |
| prerequisite 방향 오류 | “5가 필요로 하는 것이 4인가, 반대인가?” | “5가 선정되려면 4”는 `x5 <= x4`이다. |
| all-or-none을 `<=` 하나만 씀 | “동시에 포기 조건도 표현되었는가?” | `x2=x8` 또는 양방향 부등식을 쓴다. |
| mixed integer와 binary integer 혼동 | “연속 수량변수가 있는가?” | 연속과 binary가 섞이면 mixed integer다. |

## 5. 다음 Phase 연결
- P0007에서는 LP relaxation 값을 upper bound로 쓰고, integer feasible solution을 incumbent로 갱신한다.
- 반올림 대신 branch-and-bound로 정수 최적성을 증명한다.
- fixed-charge와 0-1 선택 구조는 P0008의 facility location과 set covering으로 이어진다.
